# PARC2026 06: GPU / LIBERO Validation

Google Driveを正本として、実Checkpoint＋実RLDSのGPU推論を検証します。任意で、公式LIBERO環境に予測Actionを入力する因果的な閉ループ1 Trialも実行します。

このNotebookは学習を行いません。公式Benchmarkの初期状態や観測は評価目的だけに使い、学習データへ追加しません。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/yu37330/Physical_ai.git'
BRANCH = 'main'
# Leave the repo directory first: deleting the current working directory breaks
# every later ! command with "getcwd: cannot access parent directories".
%cd /content
!rm -rf /content/Physical_ai
!git clone --branch $BRANCH $REPO_URL /content/Physical_ai
%cd /content/Physical_ai

!python scripts/check_colab_runtime.py --require-cuda


## 0. OpenVLA・RLDS依存関係を準備

現在のColab Pythonに合わせてTensorFlow系依存を選択します。Python 3.10/3.11では従来のTensorFlow 2.15系、Python 3.12ではTensorFlow 2.19系を使用します。


In [ ]:
!PROJECT_ROOT=/content/Physical_ai WORKDIR=/content/openvla-oft bash training/openvla_oft_a100/scripts/bootstrap_colab.sh
!OPENVLA_OFT_SOURCE=/content/openvla-oft bash submission/openvla_oft_offline/scripts/prepare_vendor.sh
!python -m pip install -q -r training/openvla_oft_a100/requirements-data.txt
!python scripts/check_colab_runtime.py --require-cuda


## 1. Drive上の実データを指定

Checkpointは`30_models/openvla_oft_plus`を優先し、未作成なら`30_models/openvla_oft_plus_base`へフォールバックします。  
`RLDS_DATASET_DIR`には`dataset_info.json`が存在するTFDS Builder directoryが必要です。


In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/PARC2026')
os.environ['PHYSICAL_AI_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

MODEL_ROOT = DRIVE_ROOT / '30_models'
LOCAL_MODEL_ROOT = Path('/content/work/models')
CHECKPOINT_CANDIDATES = [
    MODEL_ROOT / 'openvla_oft_plus',
    MODEL_ROOT / 'openvla_oft_plus_base',
    LOCAL_MODEL_ROOT / 'openvla_oft_plus_base',
]
CHECKPOINT_DRIVE_DIR = next(
    (path for path in CHECKPOINT_CANDIDATES if path.is_dir()),
    None,
)
CHECKPOINT_WORK_DIR = Path('/content/work/openvla_oft_plus')

RLDS_CANDIDATES = [
    DRIVE_ROOT / '20_processed/rlds/parc_libero_plus_selected/1.0.0',
    Path('/content/work/mini_rlds/parc_libero_plus_selected/1.0.0'),
]
RLDS_DATASET_DIR = next(
    (path for path in RLDS_CANDIDATES if (path / 'dataset_info.json').is_file()),
    RLDS_CANDIDATES[0],
)
GPU_REPORT = DRIVE_ROOT / '40_experiments/gpu_validation/latest_gpu_validation.json'

print('Available model directories:')
if MODEL_ROOT.is_dir():
    for path in sorted(MODEL_ROOT.iterdir()):
        print(' -', path)
else:
    print(' - drive model root is missing:', MODEL_ROOT)
if LOCAL_MODEL_ROOT.is_dir():
    for path in sorted(LOCAL_MODEL_ROOT.iterdir()):
        print(' -', path)

if CHECKPOINT_DRIVE_DIR is None:
    raise FileNotFoundError(
        'Checkpointが見つかりません。01_model_feasibility.ipynbでBase Checkpointを'
        '取得するか、CHECKPOINT_CANDIDATESへ実際のパスを追加してください。'
    )

if not RLDS_DATASET_DIR.is_dir():
    raise FileNotFoundError(
        f'RLDS Builder directoryが見つかりません: {RLDS_DATASET_DIR}\n'
        '02_dataset_prepare.ipynbで変換するか、実際のdataset_info.jsonがある'
        'ディレクトリをRLDS_CANDIDATESへ追加してください。'
    )

if not (RLDS_DATASET_DIR / 'dataset_info.json').is_file():
    raise FileNotFoundError(
        f'dataset_info.jsonがありません: {RLDS_DATASET_DIR}'
    )

CHECKPOINT_WORK_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', CHECKPOINT_DRIVE_DIR)
print('RLDS:', RLDS_DATASET_DIR)
print('Report:', GPU_REPORT)

In [ ]:
!rsync -ah --delete --info=progress2 "{CHECKPOINT_DRIVE_DIR}/" "{CHECKPOINT_WORK_DIR}/"


## 2. 実Checkpoint＋実RLDS GPU Gate

Warm-up後に複数Frameを推論し、CUDA環境、Action chunk shape、有限値、Latency、Peak VRAM、教師Actionとの誤差をJSONへ保存します。


In [ ]:
!python scripts/run_colab_gpu_validation.py \
  --checkpoint-dir "{CHECKPOINT_WORK_DIR}" \
  --dataset-dir "{RLDS_DATASET_DIR}" \
  --split val \
  --episode-offset 0 \
  --start-frame 0 \
  --num-frames 3 \
  --warmup-runs 1 \
  --output "{GPU_REPORT}"


In [ ]:
import json
gpu_report = json.loads(GPU_REPORT.read_text(encoding='utf-8'))
assert gpu_report['status'] == 'pass', gpu_report
gpu_report['summary']


## 3. 任意: LIBERO因果閉ループ1 Trial

以下は記録済みReplayではありません。PolicyのActionを`OffScreenRenderEnv`へ入力し、得られた次観測で再推論します。まずTask 0／初期状態0の1 Trialだけで接続確認します。公式評価の画像保存は標準で無効です。


In [ ]:
RUN_LIBERO_CLOSED_LOOP = False

if RUN_LIBERO_CLOSED_LOOP:
    !LIBERO_WORKDIR=/content/LIBERO OPENVLA_OFT_WORKDIR=/content/openvla-oft bash training/openvla_oft_a100/scripts/bootstrap_libero_colab.sh


In [ ]:
from datetime import datetime

if RUN_LIBERO_CLOSED_LOOP:
    RUN_ID = datetime.now().strftime('libero_spatial_t0_%Y%m%d_%H%M%S')
    !python scripts/run_libero_closed_loop.py \
      --checkpoint-dir "{CHECKPOINT_WORK_DIR}" \
      --task-suite-name libero_spatial \
      --task-id 0 \
      --init-state-id 0 \
      --seed 7 \
      --max-steps 300 \
      --run-id "{RUN_ID}"


## 判定順

1. GPU事前確認が成功する
2. CheckpointとRLDS Builder directoryが見つかる
3. GPUレポートが`pass`
4. Peak VRAMとLatencyが許容範囲
5. Action chunkが`(8, 7)`かつ有限値
6. LIBERO 1 Trialが例外なく終了
7. その後にTask数・初期状態数を増やして成功率を評価
